# Bloque 1 del Colab: arranque, verificación e inventario

Los videos se leen desde Google Drive montado. Todo lo que este notebook produce se
persiste en la raíz de proyecto, así que sobrevive al cierre de la sesión y lo encuentra
el notebook `02_roi_y_normalizacion.ipynb`.


## Paso 0 — Arranque: montar Drive y cargar el paquete `equinos`

In [ ]:
# ============================================================
# ARRANQUE — persistencia en Google Drive
# ============================================================
# Esta celda es autocontenida a propósito: corre ANTES de que el paquete `equinos` esté
# disponible, así que no puede importarlo.
#
# Resuelve en cascada de dónde sale el código:
#   1. de la copia en Drive (RAIZ/codigo/equinos)
#   2. clonando el repo, si hay un GITHUB_TOKEN en los secrets de Colab
#   3. si no, falla con instrucciones concretas
#
# El repo es privado, por eso un `git clone` sin credenciales no alcanza. El token va en los
# secrets de Colab (panel izquierdo, icono de llave) y NUNCA dentro del notebook: lo que se
# escribe en una celda queda en el archivo y se comparte con él.

import shutil
import subprocess
import sys
from pathlib import Path

NOMBRE_PROYECTO = "tesina_equinos"
REPO = "github.com/sergioyanez/Tesina-UGR.git"

# Rama del repo que Colab clona. `main` NO tiene el trabajo de la rama de ejecución local, así
# que sin esto Colab corre una versión vieja del código sin avisar: el clone funciona, los
# imports funcionan, y los resultados salen de otro pipeline.
#
# Se declara acá arriba y no dentro de la llamada a `git clone` a propósito: es lo que hay que
# cambiar para volver a `main` cuando la rama se integre, y tiene que estar a la vista.
#
# El mismo valor está duplicado en 00_pipeline.ipynb y en 01_preparar_dataset_equinos.ipynb.
# Las dos celdas de arranque son autocontenidas porque corren ANTES de que el paquete `equinos`
# exista, así que no pueden compartir la constante. Si se cambia una, cambiar la otra: dos
# versiones del código conviviendo es el peor escenario para depurar.
RAMA = "ejecucion-local-gpu"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    from google.colab import drive

    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")

    CODIGO = Path("/content/drive/MyDrive") / NOMBRE_PROYECTO / "codigo"
    CODIGO.mkdir(parents=True, exist_ok=True)

    if not (CODIGO / "equinos").is_dir():
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception as error:
            # Se distingue "el secret no existe" de "existe pero este notebook no tiene acceso".
            # El interruptor por notebook es lo que más se olvida, y sin distinguirlo el mensaje
            # manda a crear un secret que ya está creado.
            if type(error).__name__ == "NotebookAccessError":
                raise SystemExit(
                    "El secret GITHUB_TOKEN existe pero este notebook no tiene acceso.\n"
                    "  Panel izquierdo -> icono de llave -> activá el interruptor de este notebook."
                )
            token = None

        if not token:
            raise SystemExit(
                f"Falta el paquete `equinos` en {CODIGO}.\n"
                "  Opción A: cargá un secret GITHUB_TOKEN en Colab (panel izquierdo, icono de llave).\n"
                f"  Opción B: copiá a mano la carpeta `equinos/` del repo dentro de {CODIGO}."
            )

        clon = Path("/content/_repo_tesina")
        shutil.rmtree(clon, ignore_errors=True)
        resultado = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", RAMA,
             f"https://{token}@{REPO}", str(clon)],
            capture_output=True, text=True,
        )
        if resultado.returncode != 0:
            # No se imprime el comando ni stderr: contienen el token.
            raise SystemExit("Falló el clonado. Verificá que GITHUB_TOKEN tenga permiso de lectura del repo.")

        shutil.copytree(clon / "equinos", CODIGO / "equinos", dirs_exist_ok=True)
        print(f"Paquete `equinos` copiado a {CODIGO}")

    sys.path.insert(0, str(CODIGO))
else:
    sys.path.insert(0, str(Path.cwd()))

# Si `equinos` ya se importó en este kernel, Python NO vuelve a leer los archivos: se queda con
# lo que tiene en sys.modules. Una sincronización recién hecha quedaría invisible hasta reiniciar
# el entorno, y el síntoma es desconcertante — los archivos nuevos están en Drive y el import
# igual falla. Se purga el caché de módulos para que el import de abajo lea el código vigente.
for _modulo in [m for m in list(sys.modules) if m == "equinos" or m.startswith("equinos.")]:
    del sys.modules[_modulo]

from equinos import cache, catalogo, checkpoint, entorno, etapas, reporte, rutas

RAIZ = rutas.inicializar()


## Paso 0a — Parámetros de la etapa

In [ ]:
# Parámetros de la etapa.
#
# Papermill inyecta una celda justo debajo de ésta cuando el notebook se ejecuta desde
# 00_pipeline.ipynb. Ejecutado a mano, valen estos valores por defecto.
#
# DURACION_CLIP se fija en el centro del rango admitido (5-7 s). Bajarla a 5.0 o subir
# SOLAPAMIENTO genera más clips, útil dado el tamaño reducido del dataset, pero produce
# muestras más parecidas entre sí: sólo es válido si la partición agrupa por caballo
# (ver columna id_caballo).

DURACION_CLIP   = 6.0    # duración objetivo de cada clip, en segundos
DURACION_MINIMA = 5.0    # piso admitido por el plan de trabajo
DURACION_MAXIMA = 7.0    # techo admitido por el plan de trabajo
SOLAPAMIENTO    = 0.0    # fracción de solape entre clips (0.0 = sin solape)
NUM_FRAMES      = 16     # frames uniformes que se extraen por clip

# Piso para admitir ENTERO un video más corto que un clip, en segundos. None lo desactiva y
# vuelve al comportamiento anterior.
#
# Existe porque el piso de 5 s no filtraba tomas malas: filtraba ejemplares. Medido sobre el
# catálogo de 234 videos, `Disgustado` —la clase que limita todo— tiene 42 caballos y su video
# mediano dura 2,9 s, así que el piso dejaba entrar 11: los otros 31 quedaban afuera enteros.
# Como el ejemplar está correlacionado con la clase, sostener ese piso no cuidaba la calidad
# del dato, alimentaba el confounder.
#
# Lo que NO hace: tocar los videos que sí llegan al piso. Sus ventanas se calculan igual que
# antes, así que el diseño de 6 s del plan de trabajo sigue rigiendo para todo lo que puede
# cumplirlo, y los clips ya generados no cambian. La contrapartida es que el dataset pasa a
# tener clips de largo variable: 16 frames que abarcan distinto tiempo según el clip.
# Queda auditable en la columna `duracion_seg` de inventario_clips.csv.
#
# BAJADO DE 2.0 A 1.0 EL 28/08/2026, con el mismo argumento un escalón más abajo y ahora medido
# de punta a punta, no supuesto:
#
#   - La duración está correlacionada con la clase. `Disgustado` tiene mediana de 2,83 s y el
#     52,9 % de sus videos dura menos de 3 s; `Interesado` y `Sereno` tienen medianas de 8,33 y
#     10,43 s y menos del 3 % por debajo de 3 s. O sea que CUALQUIER piso de duración filtra
#     preferentemente a la clase minoritaria. No filtra tomas malas: filtra ejemplares.
#   - Con el piso en 2,0 había 11 caballos de `Disgustado` que no aportaban NINGÚN clip porque
#     todos sus videos duraban menos. Bajando a 1,0 entraron 7: 203, 205, 208, 209, 233, 234
#     y 235.
#   - Y alinean MEJOR que el resto de su clase: 68,0 % contra 63,1 %. No es casualidad — 16
#     frames sobre 1,5 s están mucho más juntos en el tiempo que sobre 6 s, hay menos movimiento
#     entre ellos y DeepLabCut pierde el hocico con menos frecuencia.
#   - Efecto medido en la métrica que decide: `Disgustado` pasó de 8 a 10 ejemplares con al
#     menos un clip íntegro, y el total de clips de la clase de 74 a 88.
#
# LO QUE HAY QUE DECLARAR EN EL INFORME, porque es el costo real: los clips que entran por este
# piso cubren una ventana temporal de 1-2 s, contra los 5-7 s del diseño del plan de trabajo.
# Sus 16 frames abarcan menos tiempo, así que NO son muestras equivalentes a las demás. Se
# distinguen por `duracion_seg` en inventario_clips.csv, y conviene reportar la métrica con y
# sin ellos antes de dar por buena cualquier conclusión.
#
# Lo que NO se cambió, y por qué: la duración objetivo del clip sigue en 6 s. Acortarla
# generaría más clips, pero beneficiaría MÁS a las clases que ya tienen material —sus videos son
# largos y se parten en más pedazos—, y la brecha `Sereno`/`Disgustado` pasaría de 2,1:1 a
# 3,6:1 con clips de 2 s. Trocear no crea ejemplares nuevos, que es lo único que rompe el
# confounder.
PISO_VIDEO_CORTO = 1.0

# Carpeta que contiene las tres carpetas de clase. Vacía = se resuelve sola (ver
# catalogo.raiz_videos): la carpeta de la raíz de proyecto si tiene clases adentro, y si no el
# acceso directo a "Videos de prueba" en Mi unidad. 00_pipeline.ipynb inyecta acá la que localizó,
# que es la compartida.
CARPETA_VIDEOS = ""

# True descarta lo ya hecho y re-ejecuta la etapa desde cero.
LIMPIAR_ETAPA   = False

# True cuando el notebook lo ejecuta 00_pipeline.ipynb. Apaga las barras de progreso: bajo
# papermill cada actualización de tqdm se guarda como una salida más del notebook, y ese
# notebook vive entero en memoria y se reserializa una y otra vez. Con 92 clips eso consume
# toda la RAM de la sesión y termina colgando Colab.
EJECUCION_AUTOMATICA = False


## Paso 0b — Limpieza explícita (opcional)

In [ ]:
# ============================================================
# LIMPIEZA EXPLÍCITA (desactivada por defecto)
# ============================================================
# La versión anterior de este paso borraba clips, frames y resultados en CADA ejecución.
# Eso volvía imposible reanudar nada: reabrir el notebook destruía el trabajo de las etapas
# siguientes. Ahora el borrado es opt-in y dirigido a esta etapa.

if LIMPIAR_ETAPA:
    etapas.limpiar("preparar_dataset")
else:
    etapas.diagnostico()

def progreso(iterable, total, etiqueta):
    """tqdm cuando hay alguien mirando; una línea cada 10 unidades cuando ejecuta el pipeline."""
    if not EJECUCION_AUTOMATICA:
        from tqdm import tqdm
        yield from tqdm(iterable, total=total, desc=etiqueta)
        return

    import time as _time
    arranque = _time.time()
    for i, elemento in enumerate(iterable, 1):
        yield elemento
        if i % 10 == 0 or i == total:
            print(f"  {etiqueta}: {i}/{total} · {(_time.time() - arranque) / 60:.1f} min", flush=True)


## Paso 1 — Instalar dependencias


In [ ]:
# ============================================================
# DEPENDENCIAS
# ============================================================
# En Colab la máquina es efímera y hay que instalar en cada sesión. En local el entorno está
# construido de antemano y pinneado en `requirements-local.txt`, así que esta celda NO instala:
# `pip install` acá le cambiaría las versiones al intérprete que ejecuta el notebook —que fuera
# de Colab es una máquina de trabajo real, no una sesión descartable— y `deeplabcut` fija
# `numpy<2`, o sea que se lleva puesto todo lo compilado contra numpy 2.
#
# Fuera de Colab, entonces, se verifica. Un entorno incompleto tiene que fallar acá, en el
# arranque, y no cuarenta minutos después con un traceback que apunta a otra cosa.

if EN_COLAB:
    !pip install -q opencv-python pandas matplotlib tqdm scikit-learn

entorno.verificar("basica")


## Paso 2 — Localizar los videos en Drive

In [ ]:
# ============================================================
# LOCALIZAR LOS VIDEOS EN DRIVE
# ============================================================
# Antes se descargaban con gdown en cada sesión (~1.9 GB). Ahora se leen montados: los videos
# ya viven en Drive, no hace falta traerlos a ningún lado.
#
# La carpeta que las contiene se resuelve en cascada y NO se da por supuesta: `raiz_videos`
# exige que el candidato tenga carpetas de clase adentro. Una carpeta que existe pero está vacía
# produciría un inventario en cero y tres etapas procesando nada sin fallar.

videos_dir = catalogo.raiz_videos(CARPETA_VIDEOS)

print("Carpeta de videos:", videos_dir)
print("Clases declaradas:", ", ".join(catalogo.SUBCARPETAS_POR_CLASE))
print("Carpetas presentes:", sorted(p.name for p in videos_dir.iterdir() if p.is_dir()))


## Paso 3 — Verificar la estructura de carpetas

In [ ]:
# Sólo las carpetas de clase y lo que cuelga de ellas: la base puede tener material vecino que
# no es del dataset, y listarlo entero sobre Drive montado es una llamada de red por entrada.

print("Contenido de las carpetas de clase:")

for etiqueta in catalogo.SUBCARPETAS_POR_CLASE:
    carpeta_clase = catalogo.carpeta_de_clase(videos_dir, etiqueta)
    if carpeta_clase is None:
        print(f"\n  {etiqueta}: SIN CARPETA en {videos_dir}")
        continue
    print(f"\n  {carpeta_clase.name}/")
    for ruta in sorted(carpeta_clase.rglob("*")):
        marca = "/" if ruta.is_dir() else ""
        print(f"    {ruta.relative_to(carpeta_clase)}{marca}")


## Paso 4 — Buscar videos

In [ ]:
# ============================================================
# CATÁLOGO DE VIDEOS POR CLASE
# ============================================================
# La clase de un video ya NO sale del nombre de su carpeta contenedora. Salía de ahí, y por eso
# los 10 videos de `Disgustado` que viven en `Seleccionar cabeza?` quedaban afuera del dataset
# —justo la clase minoritaria, la que más falta hace— mientras que una carpeta de descarga
# llamada `Sereno-20260819T000125Z-1-001/Sereno` entraba entera y sin aviso.
#
# Ahora cada clase declara qué subcarpetas suman, en `equinos/catalogo.py`, y no son las mismas
# en todas: `Disgustado` suma dos, `Interesado` y `Sereno` una. Lo que no está declarado queda
# afuera, pero se informa con nombre y conteo — descartar y perder no son lo mismo.

inventario = catalogo.informe(catalogo.inventariar(videos_dir))

# Corta acá si el catálogo quedó vacío o si hay nombres de archivo repetidos. El nombre es la
# clave del checkpoint y la base del nombre de cada clip, que es la clave de unión con el
# notebook 02: un choque mezclaría frames de dos clases bajo la misma clave, sin error visible.
catalogo.verificar(inventario)

videos_por_clase = inventario.videos

# Lista plana para las etapas siguientes, con la clase a mano: no se puede recuperar del padre
# de la ruta cuando el video vino de una subcarpeta.
videos = [ruta for encontrados in videos_por_clase.values() for ruta in encontrados]
clase_de = {ruta: etiqueta
            for etiqueta, encontrados in videos_por_clase.items()
            for ruta in encontrados}


## Paso 5 — Crear inventario técnico de videos

In [ ]:
import cv2
import pandas as pd


def abrir_video(ruta):
    """VideoCapture con la orientación de los metadatos ya aplicada.

    Los videos de celular guardan la rotación como metadato en vez de rotar los píxeles, y
    OpenCV entrega el frame como está almacenado salvo que se le pida lo contrario. Medido en
    Colab (OpenCV 4.11) sobre un video con rotación 90: sin la bandera el frame sale 1920x1080
    acostado; con la bandera, 1080x1920 derecho.

    El costo de no hacerlo fue grande y silencioso: el 19/08/2026, los 13 videos rotados del
    dataset perdieron el 55,6 % de sus frames —el detector no reconoce un caballo girado 90
    grados, y sin caja no hay recorte— contra 0,0 % de los 49 derechos. Y los frames rotados
    que sí pasaban llegaban acostados al modelo de pose, que es por qué los ejemplares dueños
    de esos videos estaban todos al fondo de la tabla de cobertura.

    TODA lectura de video del pipeline tiene que pasar por acá.
    """
    cap = cv2.VideoCapture(str(ruta))
    cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)
    return cap


datos = []

for video_path in videos:
    cap = abrir_video(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duracion = frames / fps if fps and fps > 0 else 0

    cap.release()

    datos.append({
        "archivo": video_path.name,
        # La etiqueta sale del catálogo, no del nombre de la carpeta que contiene el archivo:
        # para un video de `Disgustado/Seleccionar cabeza?` esa carpeta no es la clase.
        "clase": clase_de[video_path],
        # La carpeta real de la que salió, que ahora puede ser una subcarpeta. Es lo que permite
        # rastrear después de dónde vino cada muestra sin volver a recorrer Drive.
        "carpeta": video_path.parent.name,
        # Relativa a la raíz de proyecto: así el CSV sigue siendo legible en otra sesión,
        # en otra cuenta de Drive o en la máquina local.
        "ruta": rutas.relativizar(video_path),
        "fps": fps,
        "frames": frames,
        "duracion_seg": duracion,
        "ancho": ancho,
        "alto": alto
    })

df_videos = pd.DataFrame(datos)
df_videos


## Paso 6 — Conteo por clase


In [ ]:
# La etiqueta, y aparte de qué carpeta salió cada video: con las subcarpetas sumando al
# catálogo, las dos cosas dejaron de ser la misma.
print(df_videos["clase"].value_counts().to_string())
print()
print(df_videos.groupby(["clase", "carpeta"]).size().to_string())


## Paso 7 — Guardar inventario

In [ ]:
inventario_path = rutas.carpeta("resultados") / "inventario_videos.csv"

df_videos.to_csv(inventario_path, index=False)

print("Inventario guardado en:")
print(inventario_path)


## Paso 8 — Mostrar una muestra de video

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_ejemplo = videos[0]

# Por abrir_video y no cv2.VideoCapture: si la vista previa mostrara el frame acostado,
# el problema de orientación se vería acá y no 3 horas después como frames faltantes.
cap = abrir_video(video_ejemplo)
ret, frame = cap.read()
cap.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 5))
    plt.imshow(frame_rgb)
    plt.axis("off")
    plt.title(video_ejemplo.name)
    plt.show()
else:
    print("No se pudo leer el video:", video_ejemplo)

## Bloque 2: Segmentación en clips de 5 a 7 segundos

El plan de trabajo define como unidad de análisis un clip de **5 a 7 segundos**.
Los videos originales duran entre 8 y 37 segundos, así que este bloque los recorta
en ventanas dentro de ese rango. Además de cumplir con la especificación, la
segmentación multiplica la cantidad de muestras disponibles, algo crítico con
sólo 34 videos de origen.

Paso 9  — Configuración y funciones de segmentación

Paso 10 — Recortar los videos en clips

Paso 11 — Verificar duraciones y conteo por clase

Paso 12 — Guardar inventario_clips.csv

## Paso 9 — Funciones de segmentación

In [ ]:
import re

import cv2
import numpy as np

# ============================================================
# FUNCIONES DE SEGMENTACIÓN EN CLIPS
# ============================================================
# Los parámetros (DURACION_CLIP y compañía) están en la celda de parámetros del Paso 0.

clips_dir = rutas.carpeta("clips")


def normalizar_nombre(texto):
    """
    Reemplaza espacios y caracteres problemáticos para usar el texto
    como nombre de archivo. Varios videos originales tienen espacios
    (por ejemplo "037_Copia de Box 7-3.mp4").
    """
    return re.sub(r"[^A-Za-z0-9_-]+", "_", texto).strip("_")


def obtener_id_caballo(nombre_archivo):
    """
    Extrae el prefijo numérico del nombre del archivo, que identifica al ejemplar.
    Ejemplo: "037_box-7-1_28oDPfEr.mp4" -> "037"

    Es la clave de agrupamiento para particionar los datos sin fuga entre
    conjuntos: varios videos comparten caballo, y todos los clips y frames
    de un mismo caballo deben caer del mismo lado del split.
    """
    coincidencia = re.match(r"^(\d+)", nombre_archivo)
    return coincidencia.group(1) if coincidencia else "desconocido"


def calcular_ventanas(total_frames, fps):
    """
    Calcula los rangos [inicio, fin) de frames de cada clip de un video.

    Recorre el video en ventanas de DURACION_CLIP segundos. Si el resto que
    quedaría suelto al final es demasiado corto para formar su propio clip,
    se absorbe en la ventana anterior mientras no se supere DURACION_MAXIMA;
    en caso contrario se descarta.

    Un video más corto que DURACION_MINIMA no admite ninguna ventana. En vez de perderlo,
    entra entero como un único clip si dura al menos PISO_VIDEO_CORTO. La regla se aplica
    sólo cuando no salió ninguna ventana: para los videos que llegan al piso, esto no cambia
    absolutamente nada.
    """
    if not fps or fps <= 0 or total_frames <= 0:
        return []

    frames_clip = int(round(DURACION_CLIP * fps))
    frames_minimos = int(round(DURACION_MINIMA * fps))
    frames_maximos = int(round(DURACION_MAXIMA * fps))
    paso = max(1, int(round(frames_clip * (1 - SOLAPAMIENTO))))

    ventanas = []
    inicio = 0

    while inicio + frames_minimos <= total_frames:
        fin = min(inicio + frames_clip, total_frames)

        restante = total_frames - fin
        if 0 < restante < frames_minimos:
            fin = min(total_frames, inicio + frames_maximos)

        ventanas.append((inicio, fin))
        inicio += paso

    if not ventanas and PISO_VIDEO_CORTO:
        if total_frames >= int(round(PISO_VIDEO_CORTO * fps)):
            ventanas.append((0, total_frames))

    return ventanas


print("Duración objetivo por clip:", DURACION_CLIP, "segundos")
print("Rango admitido:", DURACION_MINIMA, "a", DURACION_MAXIMA, "segundos")
print("Video corto admitido entero desde:", PISO_VIDEO_CORTO, "segundos")
print("Carpeta de salida:", clips_dir)


## Paso 10 — Recortar los videos en clips (reanudable)

In [ ]:
import pandas as pd
from tqdm import tqdm

# ============================================================
# RECORTE DE VIDEOS EN CLIPS  (reanudable)
# ============================================================
# Escribir ~2.7 GB de clips a Drive lleva varios minutos y la sesión de Colab puede cortarse
# en el medio. El checkpoint es por VIDEO: al relanzar, se procesan sólo los que faltan.
#
# Cambiar cualquier parámetro de segmentación invalida el checkpoint entero. Mezclar clips
# generados con dos configuraciones distintas daría un dataset inconsistente que no falla ni
# avisa: simplemente arrastra el error hasta las métricas finales.

CLAVES_SEGMENTACION = ["DURACION_CLIP", "DURACION_MINIMA", "DURACION_MAXIMA", "SOLAPAMIENTO",
                       "PISO_VIDEO_CORTO"]

cp = checkpoint.abrir("preparar_dataset", globals(), CLAVES_SEGMENTACION)

pendientes = cp.pendientes(df_videos["archivo"].tolist())
print(f"Videos pendientes: {len(pendientes)} de {len(df_videos)}\n")

por_procesar = df_videos[df_videos["archivo"].isin(pendientes)]

for _, row in progreso(por_procesar.iterrows(), len(por_procesar), "Clips"):
    video_path = rutas.resolver(row["ruta"])
    clase = row["clase"]
    fps = row["fps"]
    total_frames = int(row["frames"])
    ancho = int(row["ancho"])
    alto = int(row["alto"])

    ventanas = calcular_ventanas(total_frames, fps)

    if not ventanas:
        print("Video sin clips válidos (demasiado corto):", video_path.name)
        cp.marcar(row["archivo"])
        continue

    salida_clase_dir = clips_dir / clase
    salida_clase_dir.mkdir(parents=True, exist_ok=True)

    cap = abrir_video(video_path)
    base = normalizar_nombre(video_path.stem)

    registros_video = []

    for i, (inicio, fin) in enumerate(ventanas):
        nombre_clip = f"{base}_clip_{i:02d}.mp4"
        clip_path = salida_clase_dir / nombre_clip

        # El escritor se crea con la forma del PRIMER FRAME REAL, no con las dimensiones del
        # inventario. VideoWriter.write() descarta en silencio los frames cuyo tamaño no
        # coincide con el declarado —sin excepción y sin valor de retorno—, así que confiar en
        # una dimensión de segunda mano convierte cualquier desajuste en un clip vacío que
        # nadie nota hasta mucho después.
        escritor = None

        cap.set(cv2.CAP_PROP_POS_FRAMES, inicio)
        frames_escritos = 0

        for _ in range(fin - inicio):
            ret, frame = cap.read()
            if not ret:
                break
            if escritor is None:
                alto_frame, ancho_frame = frame.shape[:2]
                if (ancho_frame, alto_frame) != (ancho, alto):
                    print(f"  {video_path.name}: el frame es {ancho_frame}x{alto_frame} y el "
                          f"inventario dice {ancho}x{alto}; se usa el del frame.")
                escritor = cv2.VideoWriter(
                    str(clip_path),
                    cv2.VideoWriter_fourcc(*"mp4v"),
                    fps,
                    (ancho_frame, alto_frame)
                )
            escritor.write(frame)
            frames_escritos += 1

        if escritor is not None:
            escritor.release()

        registros_video.append({
            "clase": clase,
            "id_caballo": obtener_id_caballo(video_path.name),
            "video_origen": video_path.name,
            "clip": nombre_clip,
            "ruta_clip": rutas.relativizar(clip_path),
            "frame_inicio": inicio,
            "frame_fin": inicio + frames_escritos,
            "frames": frames_escritos,
            "fps": fps,
            "duracion_seg": frames_escritos / fps if fps else 0
        })

    cap.release()

    # Las filas se anexan y recién después se marca el video como hecho: si el proceso muere
    # en el medio, lo peor que pasa es reprocesar un video.
    cp.marcar(row["archivo"], registros_video)

# id_caballo se relee como texto a propósito: pandas lo inferiría como entero y "031" pasaría
# a ser 31, rompiendo la correspondencia con el nombre del archivo de origen.
df_clips = cp.filas(dtype={"id_caballo": str})

print("\nVideos de origen:", len(df_videos))
print("Clips generados:", len(df_clips))
df_clips.head()


## Paso 11 — Verificar duraciones y conteo por clase

In [ ]:
# ============================================================
# CONTROL DE CALIDAD DE LA SEGMENTACIÓN
# ============================================================

print("Distribución de la duración de los clips (segundos):")
print(df_clips["duracion_seg"].describe())

# Ningún clip debería quedar fuera del rango que fija el plan de trabajo.
# Se admite un margen de 0.1 s por el redondeo de frames a segundos.
fuera_de_rango = df_clips[
    (df_clips["duracion_seg"] < DURACION_MINIMA - 0.1) |
    (df_clips["duracion_seg"] > DURACION_MAXIMA + 0.1)
]

print("\nClips fuera del rango de", DURACION_MINIMA, "a", DURACION_MAXIMA, "s:",
      len(fuera_de_rango))

if len(fuera_de_rango) > 0:
    print(fuera_de_rango[["clip", "duracion_seg"]])

print("\nClips por clase:")
print(df_clips["clase"].value_counts())

print("\nClips por caballo (clave de agrupamiento para el split):")
print(df_clips["id_caballo"].value_counts())

# Comparación contra el punto de partida: cuánto creció el dataset
comparacion = pd.DataFrame({
    "videos": df_videos["clase"].value_counts(),
    "clips": df_clips["clase"].value_counts()
})
comparacion["factor"] = (comparacion["clips"] / comparacion["videos"]).round(2)

print("\nVideos vs clips por clase:")
print(comparacion)

segundos_origen = df_videos["duracion_seg"].sum()
segundos_clips = df_clips["duracion_seg"].sum()

print("\nSegundos de video original:", round(segundos_origen, 1))
print("Segundos conservados en clips:", round(segundos_clips, 1))
print("Aprovechamiento:", round(100 * segundos_clips / segundos_origen, 1), "%")

## Paso 12 — Guardar inventario de clips

In [ ]:
clips_csv_path = rutas.carpeta("resultados") / "inventario_clips.csv"

df_clips.to_csv(clips_csv_path, index=False)

print("Inventario de clips guardado en:")
print(clips_csv_path)


## Bloque 3: Extracción de frames

Los frames ya no se extraen del video completo sino **de cada clip**, que es la
unidad de análisis definida en el plan de trabajo. Cada registro arrastra
`id_caballo` y `video_origen` para poder particionar los datos más adelante sin
fuga entre conjuntos.

Paso 13 — Extraer N frames uniformes por clip

Paso 14 — Conteo de frames por clase

Paso 15 — Guardar dataset_frames.csv

Paso 16 — Ver muestras de frames por clase

Pendiente (Bloque 4):

Paso 17 — Dataset PyTorch con particionado por caballo

Paso 18 — Modelo base

Paso 19 — Entrenamiento

Paso 20 — Evaluación (F1 macro y balanced accuracy)

Paso 21 — Matriz de confusión

Paso 22 — Guardar modelo

## Paso 13 — Extraer frames uniformes por clip

In [ ]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

# ============================================================
# EXTRACCIÓN DE FRAMES
# ============================================================
# Los frames van al área de trabajo local, NO a Drive: son ~1472 archivos chicos y Drive
# montado tiene una latencia por archivo que domina el tiempo total. Se persisten después,
# empaquetados en un único zip (Paso 15).

frames_dir = rutas.dir_trabajo("frames_equinos")

registros_frames = []


def extraer_frames_uniformes(clip_path, salida_dir, num_frames=16):
    """
    Extrae una cantidad fija de frames distribuidos uniformemente a lo largo del clip.
    """
    cap = abrir_video(clip_path)

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < 12:
        cap.release()
        return []

    actual_num_frames = min(total_frames, num_frames)
    indices = np.linspace(0, total_frames - 1, actual_num_frames).astype(int)

    frames_guardados = []

    for i, frame_idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()

        if not ret:
            continue

        nombre_frame = f"frame_{i:03d}.jpg"
        frame_path = salida_dir / nombre_frame

        cv2.imwrite(str(frame_path), frame)
        frames_guardados.append(frame_path)

    cap.release()
    return frames_guardados


for _, row in progreso(df_clips.iterrows(), len(df_clips), "Frames"):
    clip_path = rutas.resolver(row["ruta_clip"])
    clase = row["clase"]
    nombre_clip = clip_path.stem

    salida_clip_dir = frames_dir / clase / nombre_clip
    salida_clip_dir.mkdir(parents=True, exist_ok=True)

    frames = extraer_frames_uniformes(
        clip_path=clip_path,
        salida_dir=salida_clip_dir,
        num_frames=NUM_FRAMES
    )

    for frame_path in frames:
        registros_frames.append({
            "clase": clase,
            "id_caballo": row["id_caballo"],
            "video_origen": row["video_origen"],
            "clip": row["clip"],
            "frame": frame_path.name,
            # Relativa a frames_dir, que es donde se expande la caché en cualquier sesión.
            "ruta_frame": frame_path.relative_to(frames_dir).as_posix(),
            "ruta_clip": row["ruta_clip"]
        })

df_frames = pd.DataFrame(registros_frames)

print("Frames extraídos:", len(df_frames))
df_frames.head()


## Paso 14 — Conteo de frames por clase

In [ ]:
df_frames["clase"].value_counts()

## Paso 15 — Guardar el CSV de frames y persistir la caché

In [ ]:
# ============================================================
# PERSISTIR EL CSV Y LA CACHÉ DE FRAMES
# ============================================================
# El zip es lo que hace que la etapa quede "completa": sin él, los frames se pierden al
# cerrarse la sesión y habría que volver a extraerlos.

frames_csv_path = rutas.carpeta("resultados") / "dataset_frames.csv"
df_frames.to_csv(frames_csv_path, index=False)

print("CSV de frames guardado en:", frames_csv_path)

cache.empaquetar(frames_dir, "frames_equinos")


## Paso 16 — Ver muestras de frames por clase

In [ ]:
import matplotlib.pyplot as plt
import cv2

clases = sorted(df_frames["clase"].unique())

for clase in clases:
    muestra = df_frames[df_frames["clase"] == clase].sample(1, random_state=42)
    ruta = frames_dir / muestra.iloc[0]["ruta_frame"]

    img = cv2.imread(str(ruta))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.title(f"Clase: {clase}")
    plt.axis("off")
    plt.show()
